# RAG(Retrieval-Augmented Generation)

## RAG란?

RAG(Retrieval-Augmented Generation)는 **검색(Retrieval)** 과 **생성(Generation)** 을 결합한 AI 기술이다.

기존 LLM(Large Language Model)은 학습된 데이터만을 기반으로 답변을 생성한다. 따라서 최신 정보나 학습되지 않은 문서에 대해서는 정확한 답변을 제공하기 어렵다.

이러한 한계를 해결하기 위해 등장한 기술이 RAG이다.

RAG는 사용자의 질문과 관련된 문서를 먼저 검색한 후(Search), 검색된 문서를 참고하여 답변을 생성(Generation)한다.

즉,

 **"검색한 정보를 기반으로 답변을 생성하는 기술"**

이라고 이해하면 된다.

---

# RAG가 필요한 이유

LLM은 매우 뛰어난 언어 생성 능력을 가지고 있지만 다음과 같은 한계가 있다.

- 최신 정보를 알지 못할 수 있다.
- 회사 내부 문서를 학습하지 않았다.
- 특정 분야의 전문 지식이 부족할 수 있다.
- 존재하지 않는 내용을 사실처럼 생성하는 Hallucination(환각) 현상이 발생할 수 있다.

예를 들어,

```
우리 회사의 휴가 규정을 알려줘.
```

라는 질문을 한다면,

LLM은 회사 내부 문서를 학습하지 않았기 때문에 정확한 답변을 생성할 수 없다.

하지만 RAG를 사용하면

1. 회사 규정 문서를 검색하고
2. 검색된 문서를 Context로 제공한 후
3. 해당 문서를 기반으로 답변을 생성한다.

따라서 더욱 신뢰성 높은 답변을 생성할 수 있다.

---

# RAG의 전체 동작 과정

RAG는 일반적으로 다음과 같은 순서로 동작한다.

```
사용자 질문
      │
      ▼
문서 검색(Retrieval)
      │
      ▼
관련 문서(Context)
      │
      ▼
Prompt 생성
      │
      ▼
     LLM
      │
      ▼
최종 답변 생성
```

---

# RAG 구성 요소

RAG 시스템은 크게 다섯 가지 요소로 구성된다.

## 1. Documents

검색의 대상이 되는 원본 데이터이다.

예를 들어

- PDF
- Word
- TXT
- CSV
- 데이터베이스(DB)
- 웹 페이지
- 사내 문서

등이 모두 Documents가 될 수 있다.

---

## 2. Embedding Model

Embedding Model은 텍스트를 숫자(Vector)로 변환하는 모델이다.

예를 들어

```
"인공지능"
```

이라는 단어를

```
[0.13, -0.27, 0.94, ...]
```

와 같은 벡터(Vector)로 변환한다.

이러한 벡터를 이용하면 단순한 키워드 검색이 아닌 의미 기반(Semantic Search)의 검색이 가능해진다.

---

## 3. Vector Database

Embedding된 벡터를 저장하는 데이터베이스이다.

대표적인 Vector Database는 다음과 같다.

- FAISS
- Chroma
- Pinecone
- Milvus
- Weaviate

이번 실습에서는 **FAISS**를 사용한다.

---

## 4. Retriever

Retriever는

 **사용자의 질문과 가장 유사한 문서를 검색하는 역할**

을 수행한다.

즉,


Question
      │
      ▼
Retriever
      │
      ▼
관련 Document


의 과정을 담당한다.

Retriever는 문서를 생성하는 것이 아니라 검색만 수행한다.

---

## 5. LLM

Retriever가 검색한 문서를 Context로 전달받아 최종 답변을 생성한다.

즉,

```
Question
+
Context
↓
LLM
↓
Answer
```

의 역할을 수행한다.

---

# Context란?

Context는 Retriever가 검색한 문서를 의미한다.

예를 들어,

사용자가

```
혼자 여행하기 좋은 곳 추천해줘.
```

라고 질문하면,

Retriever는

```
제주도는 혼자 여행하기 좋은 여행지이다.

경주는 역사 문화 도시이다.

부산은 해안 관광지로 유명하다.
```

와 같은 문서를 검색한다.

이 검색된 문서 전체가 Context가 된다.

LLM은 Context를 참고하여 답변을 생성한다.

---

# RAG 실행 흐름

실제 RAG 시스템은 다음과 같은 순서로 동작한다.

```
사용자 질문
      │
      ▼
Embedding
      │
      ▼
Vector Search
      │
      ▼
Retriever
      │
      ▼
관련 문서(Document)
      │
      ▼
Context 생성
      │
      ▼
PromptTemplate
      │
      ▼
ChatOpenAI
      │
      ▼
최종 답변
```

---

# LangChain에서의 RAG 구성

이번 실습에서는 LangChain의 LCEL(LangChain Expression Language)을 사용하여 RAG를 구현한다.

구성 요소는 다음과 같다.

```
RunnableParallel
        │
        ├───────────────┐
        ▼                              ▼
Retriever                       RunnablePassthrough
        │                              │
        ▼                              ▼
RunnableLambda                       Question
        │
        ▼
Context
        │
        └───────┐
                        ▼
               ChatPromptTemplate
                        │
                        ▼
                  ChatOpenAI
                        │
                        ▼
               StrOutputParser
                        │
                        ▼
                  Final Answer
```



# 이번 실습에서 사용할 주요 클래스

| 클래스 | 역할 |
|---------|------|
| OpenAIEmbeddings | 문서를 벡터(Vector)로 변환 |
| FAISS | 벡터를 저장하는 Vector Database |
| Retriever | 관련 문서를 검색 |
| RunnableLambda | Document를 문자열(Context)로 변환 |
| RunnableParallel | 여러 Runnable을 병렬 실행 |
| RunnablePassthrough | 사용자 질문을 그대로 전달 |
| ChatPromptTemplate | Prompt 생성 |
| ChatOpenAI | 최종 답변 생성 |
| StrOutputParser | AIMessage를 문자열로 변환 |



# 핵심 정리

RAG는 단순히 LLM에게 질문하는 방식이 아니라, 먼저 관련 문서를 검색한 후 검색된 문서를 기반으로 답변을 생성하는 기술이다.

즉,


검색(Retrieval) + 생성(Generation) = RAG


이번 실습에서는 FAISS를 이용하여 관련 문서를 검색하고, LCEL 기반 RAG Chain을 구성하여 검색된 Context를 ChatOpenAI에 전달하는 전체 과정을 구현해 본다.

In [1]:
# ------------------------------------------------------------
# [상세 설명]
# LangChain 실행을 위한 필수 패키지 설치 셀이다.
#
# 1) langchain
#    → 최신 LCEL 기반 구조 사용을 위한 핵심 패키지
#
# 2) langchain-openai
#    → ChatOpenAI / OpenAI 모델 연동
#
# 3) langchain-community
#    → 외부 통합 도구 (retriever, loader 등)
#
# 4) langchain-text-splitters
#    → 문서 분할 (RAG 필수)
#
# 추가 패키지:
# - faiss-cpu → 벡터 검색(DB)
# - tiktoken → 토큰 계산
# - python-dotenv → .env 기반 환경변수 관리
#
# 주의:
# Colab에서는 설치 순서와 버전 충돌이 중요하므로
# 반드시 상단에서 한 번에 설치하는 것이 안정적이다.
# ------------------------------------------------------------

# 전체 패키지 설치
!pip install -qU langchain langchain-openai langchain-community langchain-text-splitters

# 추가 유용한 패키지들
!pip install -qU faiss-cpu tiktoken python-dotenv

# ------------------------------------------------------------
# [상세 설명]
# 최소 실행 환경만 빠르게 구성하는 간소화 설치 명령어이다.
#
# - langchain: 핵심 프레임워크
# - langchain-openai: OpenAI 연동
# - tiktoken: 토큰 계산
#
# 빠른 실습용 / 테스트용 환경에서 사용된다.
# ------------------------------------------------------------

# 라이브러리 설치 (경량 버전)
# !pip install -q langchain langchain-openai tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.9/136.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 45.2 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get("OPENAI_API_KEY")



In [4]:
# ============================================================
# 1. 상세 설명
# ============================================================

# 이 코드는 LCEL 체인 자체를 RunnableParallel 구조로 확장할 수 있다는 개념을 설명한다.
#
# 핵심 개념:
#
# - LCEL 체인(chain1, chain2)도 하나의 Runnable이다
# - RunnableParallel에 체인을 그대로 넣어 병렬 실행 가능
# - RunnablePassthrough를 통해 입력을 country 키로 매핑
#
# 전체 흐름:
#
# 입력(dict or value)
# → RunnablePassthrough
# → PromptTemplate
# → ChatOpenAI
# → 각 chain 독립 실행
#
# 특징:
# - chain 자체가 Runnable이므로 재사용 가능
# - 동일 입력으로 서로 다른 질문을 병렬 처리
# - RAG, 비교 분석 구조의 기본 패턴

# ============================================================
# 2. 기존 코드 (원본 유지)
# ============================================================

# chain1 = (
#     {'country' : RunnablePassthrough()}
#     | PromptTemplate.from_template('{country}의 수도는?')
#     | ChatOpenAI()
# )
#
# chain2 = (
#     {'country' : RunnablePassthrough()}
#     | PromptTemplate.from_template('{country}의 면적은?')
#     | ChatOpenAI()
# )

# ============================================================
# 3. 개선 코드 (LCEL 표준 + 안정성)
# ============================================================

from langchain_core.runnables import RunnablePassthrough  # 입력 전달/매핑
from langchain_core.prompts import PromptTemplate         # 프롬프트 생성
from langchain_openai import ChatOpenAI                   # LLM 모델
from langchain_core.output_parsers import StrOutputParser # 문자열 변환

# ------------------------------------------------------------
# chain1: 수도 질문 체인
# ------------------------------------------------------------
chain1 = (
      # 입력 → country 매핑
      {"country" : RunnablePassthrough()}
      # 프롬프트 생성
      | PromptTemplate.from_template("{country}의 수도는?")
      # LLM 호출
      | ChatOpenAI(model="gpt-4o-mini", temperature=0)
      # AIMessage → 문자열 변환
      | StrOutputParser()
)

# ------------------------------------------------------------
# chain2: 면적 질문 체인
# ------------------------------------------------------------
chain2 = (
      # 입력 → country 매핑
      {"country" : RunnablePassthrough()}
      # 프롬프트 생성
      | PromptTemplate.from_template("{country}의 면적은?")
      # LLM 호출
      | ChatOpenAI(model="gpt-4o-mini", temperature=0)
      # AIMessage → 문자열 변환
      | StrOutputParser()
)

# ============================================================
# 4. 핵심 포인트
# ============================================================

# - LCEL 체인은 Runnable 자체다
# - RunnableParallel로 쉽게 병렬 확장 가능
# - 동일 입력으로 서로 다른 LLM 작업 수행
# - RAG / 비교 분석 / 멀티 QA 구조의 기본 패턴

In [5]:
# ============================================================
# 1. 상세 설명
# ============================================================

# 이 코드는 LCEL 체인(chain1, chain2)을 RunnableParallel로 결합하여
# 하나의 입력으로 여러 LLM 체인을 동시에 실행하는 구조이다.
#
# 핵심 개념:
#
# - RunnableParallel은 여러 Runnable(여기서는 chain)을 동시에 실행
# - 각 chain은 독립적인 LCEL 파이프라인으로 동작
# - 동일 입력이 각 chain으로 전달됨
# - 결과는 dict 형태로 병합되어 반환됨
#
# 입력 흐름:
#
# "대한민국"
#   → capital chain1 (수도 질문)
#   → area chain2 (면적 질문)
#   → 각각 LLM 실행
#   → 결과 merge(dict)
#
# 특징:
# - chain 자체도 Runnable이라 병렬 실행 가능
# - 구조적으로 확장성 매우 높음 (RAG / QA 비교 구조)
# - 입력은 자동으로 PromptTemplate에 매핑됨

# ============================================================
# 2. 기존 코드 (원본 유지)
# ============================================================

# combined_chain = RunnableParallel(capital=chain1, area=chain2)
# combined_chain.invoke('대한민국')

# ============================================================
# 3. 개선 코드 (LCEL 표준 실행 구조)
# ============================================================

from langchain_core.runnables import RunnableParallel  # 병렬 실행기

# ------------------------------------------------------------
# 병렬 체인 구성
# ------------------------------------------------------------
combined_chain = RunnableParallel(
    capital = chain1, # 수도 질문 체인
    area = chain2 # 면적 질문 체인
)

# ------------------------------------------------------------
# 실행
# ------------------------------------------------------------
result = combined_chain.invoke("대한민국")


print(result)

# ============================================================
# 4. 핵심 포인트
# ============================================================

# - LCEL chain = Runnable → 병렬 가능
# - RunnableParallel = multi-chain executor
# - 동일 입력 → 여러 LLM 작업 분기
# - RAG / 비교 QA / 분석 시스템 핵심 구조

{'capital': '대한민국의 수도는 서울입니다.', 'area': '대한민국의 면적은 약 100,210 평방킬로미터입니다. 이는 한반도의 남쪽 부분에 해당하며, 북한과 함께 한반도를 구성하고 있습니다.'}


In [6]:
# ============================================================
# 1. 상세 설명
# ============================================================

# 이 코드는 LangChain 기반 필수 패키지 설치 셀이다.
#
# 핵심 목적:
#
# - LangChain 최신 버전 환경 구성
# - OpenAI 연동 패키지 설치
# - RAG 및 벡터 검색 실습 준비
#
# 구성 패키지 역할:
#
# - langchain: LCEL, Runnable, 체인 프레임워크
# - langchain-openai: ChatOpenAI, OpenAI 연동
# - langchain-community: 외부 통합 도구
# - langchain-text-splitters: 문서 분할 기능
# - faiss-cpu: 벡터 DB (로컬 검색)
# - tiktoken: 토큰 계산
# - python-dotenv: 환경변수 관리 (.env)

# ============================================================
# 2. 기존 코드 (원본 유지)
# ============================================================

# !pip install -q langchain>=0.3.0 langchain-openai langchain-community langchain-text-splitters
# !pip install -q faiss-cpu tiktoken python-dotenv

# ============================================================
# 3. 개선 코드 (Colab 안정성 + 명확 버전 고정)
# ============================================================

# LangChain 최신 안정 버전 설치
!pip install -q langchain langchain-openai langchain-community langchain-text-splitters

# 벡터DB + 유틸리티 패키지 설치
!pip install -q faiss-cpu tiktoken python-dotenv

# ============================================================
# 4. 라인별 주석
# ============================================================

# langchain
# → LCEL 기반 최신 LangChain 구조 사용

# langchain-openai
# → ChatOpenAI / OpenAI API 연동

# langchain-community
# → 외부 loader, tool 통합

# langchain-text-splitters
# → 문서 chunking 처리

# faiss-cpu
# → 벡터 검색 엔진 (RAG 핵심)

# tiktoken
# → OpenAI 토큰 계산기

# python-dotenv
# → API key 환경변수 관리

# ============================================================
# 5. 핵심 포인트
# ============================================================

# - LangChain 0.3+ = LCEL 중심 구조
# - RAG 실습 필수 패키지 포함
# - Colab 환경에서 바로 실행 가능한 구성

In [7]:
# OpenAI API 키 환경 변수에 설정 (본인 키로 변경)
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get("OPENAI_API_KEY")

CharacterTextSplitter

In [8]:
# ============================================================
# 1. 상세 설명
# ============================================================

# 이 코드는 LangChain의 CharacterTextSplitter를 사용하여
# 긴 텍스트를 일정한 크기의 청크(chunk)로 분할하는 예제이다.
#
# 핵심 개념:
#
# - CharacterTextSplitter
#   → 문자열 기준으로 텍스트를 일정 길이로 분할
#
# - chunk_size
#   → 각 조각의 최대 길이
#
# - chunk_overlap
#   → 문맥 연결을 위한 중복 구간
#
# 활용:
#
# - RAG 문서 전처리
# - Vector DB 입력 데이터 생성
# - 긴 문서 embedding 최적화

# ============================================================
# 2. 기존 코드 (원본 유지)
# ============================================================

# from langchain_text_splitters import CharacterTextSplitter

# text_splitter = CharacterTextSplitter(
#     chunk_size=100,
#     chunk_overlap=10
# )

# docs = text_splitter.split_text("긴 텍스트 입력")

# ============================================================
# 3. 개선 코드 (LCEL/RAG 표준 전처리 방식)
# ============================================================

from langchain_text_splitters import CharacterTextSplitter  # 텍스트 분할기

# ------------------------------------------------------------
# 텍스트 분할기 설정
# ------------------------------------------------------------
text_splitter = CharacterTextSplitter(
     # 각 청크 최대 길이
     chunk_size = 100,
     # 앞뒤 문맥 연결용 중복 길이
     chunk_overlap = 10,
     # 줄바꿈 기준 우선 분할
     separator="\n"
)

# ------------------------------------------------------------
# 예시 텍스트
# ------------------------------------------------------------
sample_text = """
LangChain은 LLM 애플리케이션을 쉽게 만들 수 있도록 도와주는 프레임워크입니다.
이 프레임워크는 Prompt, Chain, Agent, RAG 구조를 지원합니다.
특히 LCEL을 통해 유연한 파이프라인 구성이 가능합니다.
"""

# ------------------------------------------------------------
# 텍스트 분할 실행
# ------------------------------------------------------------
docs = text_splitter.split_text(sample_text)


# 결과 출력
print(docs)

# ============================================================
# 4. 핵심 포인트
# ============================================================

# - RAG 전처리 핵심 단계
# - 문서를 embedding 가능한 단위로 분해
# - chunk_overlap은 의미 연결 유지 핵심

['LangChain은 LLM 애플리케이션을 쉽게 만들 수 있도록 도와주는 프레임워크입니다.\n이 프레임워크는 Prompt, Chain, Agent, RAG 구조를 지원합니다.', '특히 LCEL을 통해 유연한 파이프라인 구성이 가능합니다.']


In [9]:
# ============================================================
# 1. 상세 설명
# ============================================================

# 이 코드는 Colab 환경에서 텍스트 파일을 업로드하고
# LangChain의 TextLoader를 사용해 문서를 로딩하는 예제이다.
#
# 핵심 개념:
#
# - TextLoader
#   → 로컬 텍스트 파일을 Document 객체로 변환
#
# - files.upload()
#   → Colab에서 사용자 파일 업로드 기능
#
# - Document 구조
#   → page_content: 실제 텍스트
#   → metadata: 파일 정보
#
# 전체 흐름:
#
# 파일 업로드 → 경로 추출 → TextLoader 로딩 → Document 생성 → 내용 확인

# ============================================================
# 2. 기존 코드 (원본 유지)
# ============================================================

# from langchain_community.document_loaders import TextLoader
# from google.colab import files
#
# uploaded = files.upload()
# file_path = list(uploaded.keys())[0]
#
# loader = TextLoader(file_path, encoding='utf-8')
# data = loader.load()
#
# print(len(data[0].page_content))
# print(data[0].page_content[:500])

# ============================================================
# 3. 개선 코드 (Colab 안정 실행 구조)
# ============================================================

from langchain_community.document_loaders import TextLoader  # 텍스트 로더
from google.colab import files

# ------------------------------------------------------------
# 파일 업로드
# ------------------------------------------------------------
uploaded = files.upload()

# 업로드된 파일명 추출
file_path = list(uploaded.keys())[0]

# ------------------------------------------------------------
# TextLoader 생성
# ------------------------------------------------------------
loader = TextLoader(
     file_path, # 업로드된 파일 경로
     encoding= "utf-8"   # 한글 깨짐 방지
)

# ------------------------------------------------------------
# 문서 로드
# ------------------------------------------------------------
data = loader.load()

# ============================================================
# 4. 결과 확인
# ============================================================

# 전체 텍스트 길이 출력
print(len(data[0].page_content))

# 앞부분 500자 출력
print(data[0].page_content[:500])



/tmp/ipykernel_2276/2069114035.py:44: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader  # 텍스트 로더


Saving AI 최신 동향과 정책.txt to AI 최신 동향과 정책.txt
2105
AI 최신 동향과 정책 (2025년 기준)


2025년 현재, 인공지능(AI)은 기술적 성숙기를 지나 실질적인 서비스와 산업 전반으로 확산되고 있다. 단순히 텍스트를 생성하거나 이미지를 인식하는 수준을 넘어서, 인간의 복잡한 작업을 대행하는 'Agentic AI(지능형 에이전트)'가 주목받고 있다. 이러한 AI는 사용자 지시에 따라 이메일을 작성하고 회의 일정을 조율하며, 업무 흐름 전반을 자동화하는 수준에 이르렀다.

AI 모델은 점차 전문화되고 있으며, 텍스트·이미지·음성·영상 등 다양한 데이터를 동시에 처리할 수 있는 '멀티모달 AI'가 보편화되고 있다. 특히 의료, 금융, 법률 등 전문 영역에 특화된 AI가 등장함에 따라, 범용 AI를 넘는 깊이 있는 서비스를 가능하게 하고 있다.

또한, AI를 클라우드 서버가 아닌 스마트폰이나 IoT 기기 등 ‘단말기’에서 직접 작동시키는 '온디바이스 AI'와 '엣지 컴퓨팅' 기술이 확산되고 있다. 이 기술은 속도와 프라이버시 측면에서 


In [10]:
# ============================================================
# 1. 상세 설명
# ============================================================

# 이 코드는 CharacterTextSplitter를 사용하여
# 문서를 문자 단위 기준으로 청크로 분할하는 예제이다.
#
# 핵심 개념:
#
# - separator=''
#   → 문자 단위로 분할 (가장 작은 단위 split)
#
# - chunk_size=500
#   → 하나의 chunk 최대 길이
#
# - chunk_overlap=100
#   → chunk 간 문맥 유지용 겹침 구간
#
# - length_function=len
#   → 문자열 길이 계산 기준 지정
#
# 활용:
#
# - RAG에서 초미세 단위 분할
# - 특수 분석 / 문자 기반 처리
# - 긴 텍스트 정밀 segmentation

# ============================================================
# 2. 기존 코드 (원본 유지)
# ============================================================

# from langchain_text_splitters import CharacterTextSplitter
#
# text_splitter = CharacterTextSplitter(
#     separator = '',
#     chunk_size = 500,
#     chunk_overlap  = 100,
#     length_function = len,
# )
#
# texts = text_splitter.split_text(data[0].page_content)
#
# len(texts)

# ============================================================
# 3. 개선 코드 (Colab 실행 + RAG 표준 구조)
# ============================================================

from langchain_text_splitters import CharacterTextSplitter  # 문자 기반 splitter

# ------------------------------------------------------------
# CharacterTextSplitter 설정
# ------------------------------------------------------------
text_splitter = CharacterTextSplitter(
     # 문자 단위 분할
     separator="",
     # 최대 500자
     chunk_size = 500,
     # 100자 중복 유지
     chunk_overlap = 100,
     # 길이 계산 함수
     length_function = len
)

# ------------------------------------------------------------
# 텍스트 분할 실행
# ------------------------------------------------------------
texts = text_splitter.split_text(
     # TextLoader에서 로드된 문서 내용
     data[0].page_content
)

# ------------------------------------------------------------
# 결과 확인
# ------------------------------------------------------------
print(len(texts))

# ============================================================
# 4. 라인별 주석
# ============================================================

# CharacterTextSplitter
# → 텍스트를 문자 기준으로 분할하는 LangChain 전처리 도구

# separator=""
# → 문자 단위로 하나씩 split

# chunk_size=500
# → 한 조각 최대 500자 제한

# chunk_overlap=100
# → 문맥 유지 위해 100자 겹침

# length_function=len
# → 문자열 길이 기준 정의

# split_text()
# → 문자열 → 리스트(chunk list) 변환

# len(texts)
# → 생성된 chunk 개수 확인

# ============================================================
# 5. 핵심 포인트
# ============================================================

# - RAG 전처리 단계 핵심 구성 요소
# - chunk_overlap은 검색 품질에 직접 영향
# - 너무 작은 chunk_size는 의미 손실 발생

6


In [11]:
# ============================================================
# 1. 상세 설명
# ============================================================

# 이 코드는 CharacterTextSplitter로 분할된 결과 중
# 첫 번째 chunk의 길이를 확인하는 코드이다.
#
# 핵심 개념:
#
# - texts: split_text() 결과 리스트 (chunk들의 모음)
# - texts[0]: 첫 번째 chunk
# - len(): 해당 chunk의 문자 길이 계산
#
# 활용:
#
# - chunk size 검증
# - RAG 전처리 결과 점검
# - 데이터 분포 분석

# ============================================================
# 2. 기존 코드 (원본 유지)
# ============================================================

# len(texts[0])

# ============================================================
# 3. 개선 코드 (검증 목적 명확화)
# ============================================================

# 첫 번째 chunk 길이 확인
first_chunk_length = len(texts[0])

print(first_chunk_length)

# ============================================================
# 4. 라인별 주석
# ============================================================

# texts[0]
# → 첫 번째 분할된 텍스트 chunk

# len(texts[0])
# → 해당 chunk의 문자 길이 계산

# print(...)
# → 결과 출력 (디버깅/검증 목적)

# ============================================================
# 5. 핵심 포인트
# ============================================================

# - chunk 품질 확인용 기본 디버깅 코드
# - RAG에서는 chunk 길이 균일성이 중요
# - 너무 짧거나 길면 검색 성능 저하 발생


499


In [12]:
texts[0]


"AI 최신 동향과 정책 (2025년 기준)\n\n\n2025년 현재, 인공지능(AI)은 기술적 성숙기를 지나 실질적인 서비스와 산업 전반으로 확산되고 있다. 단순히 텍스트를 생성하거나 이미지를 인식하는 수준을 넘어서, 인간의 복잡한 작업을 대행하는 'Agentic AI(지능형 에이전트)'가 주목받고 있다. 이러한 AI는 사용자 지시에 따라 이메일을 작성하고 회의 일정을 조율하며, 업무 흐름 전반을 자동화하는 수준에 이르렀다.\n\nAI 모델은 점차 전문화되고 있으며, 텍스트·이미지·음성·영상 등 다양한 데이터를 동시에 처리할 수 있는 '멀티모달 AI'가 보편화되고 있다. 특히 의료, 금융, 법률 등 전문 영역에 특화된 AI가 등장함에 따라, 범용 AI를 넘는 깊이 있는 서비스를 가능하게 하고 있다.\n\n또한, AI를 클라우드 서버가 아닌 스마트폰이나 IoT 기기 등 ‘단말기’에서 직접 작동시키는 '온디바이스 AI'와 '엣지 컴퓨팅' 기술이 확산되고 있다. 이 기술은 속도와 프라이버시 측면에서"

In [13]:
# 줄바꿈 문자를 기준으로 분할

text_splitter = CharacterTextSplitter(
    separator= "\n",
    chunk_size = 500,
    chunk_overlap = 100,
    length_function = len

)

texts = text_splitter.split_text(data[0].page_content)

len(texts)


5

In [14]:
len(texts[0]), len(texts[1]), len(texts[2])


(387, 494, 392)

In [15]:
texts[0]


"AI 최신 동향과 정책 (2025년 기준)\n2025년 현재, 인공지능(AI)은 기술적 성숙기를 지나 실질적인 서비스와 산업 전반으로 확산되고 있다. 단순히 텍스트를 생성하거나 이미지를 인식하는 수준을 넘어서, 인간의 복잡한 작업을 대행하는 'Agentic AI(지능형 에이전트)'가 주목받고 있다. 이러한 AI는 사용자 지시에 따라 이메일을 작성하고 회의 일정을 조율하며, 업무 흐름 전반을 자동화하는 수준에 이르렀다.\nAI 모델은 점차 전문화되고 있으며, 텍스트·이미지·음성·영상 등 다양한 데이터를 동시에 처리할 수 있는 '멀티모달 AI'가 보편화되고 있다. 특히 의료, 금융, 법률 등 전문 영역에 특화된 AI가 등장함에 따라, 범용 AI를 넘는 깊이 있는 서비스를 가능하게 하고 있다."

RecursiveCharacterTextSplitter

In [16]:
# ============================================================
# 1. 상세 설명
# ============================================================

# 이 코드는 RecursiveCharacterTextSplitter를 사용하여
# 문서를 의미 단위를 고려해 재귀적으로 분할하는 예제이다.
#
# CharacterTextSplitter와 차이:
#
# - CharacterTextSplitter
#   → 단순 문자/구분자 기반 분할
#
# - RecursiveCharacterTextSplitter
#   → 문단 → 문장 → 단어 → 문자 순으로 "의미 유지 우선" 분할
#
# 핵심 개념:
#
# - chunk_size=500
#   → 최대 500자 기준
#
# - chunk_overlap=100
#   → 문맥 유지용 중복 영역
#
# - length_function=len
#   → 길이 기준 함수
#
# 활용:
#
# - RAG에서 가장 표준적인 splitter
# - 의미 보존이 중요한 문서 처리
# - 검색 정확도 향상

# ============================================================
# 2. 기존 코드 (원본 유지)
# ============================================================

# from langchain.text_splitter import RecursiveCharacterTextSplitter
#
# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size = 500,
#     chunk_overlap  = 100,
#     length_function = len,
# )
#
# texts = text_splitter.split_text(data[0].page_content)
#
# len(texts)

# ============================================================
# 3. 개선 코드 (RAG 표준 + 최신 LangChain 구조)
# ============================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter  # 최신 권장 import

# ------------------------------------------------------------
# RecursiveCharacterTextSplitter 설정
# ------------------------------------------------------------
text_splitter = RecursiveCharacterTextSplitter(
     # 최대 chunk 크기
     chunk_size = 500,
     # 문맥 유지 overlap
     chunk_overlap = 100,
     # 길이 기준 함수
     length_function = len
)

# ------------------------------------------------------------
# 문서 분할 실행
# ------------------------------------------------------------
texts = text_splitter.split_text(
     # TextLoader 결과
     data[0].page_content
)

# ------------------------------------------------------------
# 결과 확인
# ------------------------------------------------------------
print(len(texts))

# ============================================================
# 4. 라인별 주석
# ============================================================

# RecursiveCharacterTextSplitter
# → 의미 기반(재귀적 구조) 텍스트 분할기

# chunk_size
# → 최대 500자 기준 분할

# chunk_overlap
# → 문맥 유지용 100자 중복

# split_text()
# → 긴 텍스트 → chunk 리스트 변환

# len(texts)
# → 생성된 chunk 개수 확인

# ============================================================
# 5. 핵심 포인트
# ============================================================

# - CharacterSplitter보다 훨씬 "실무 RAG 표준"
# - 문단 구조를 유지하며 자연스럽게 분할
# - 검색 정확도 향상에 매우 중요

5


In [17]:
len(texts[0]), len(texts[1]), len(texts[2])


(390, 497, 393)

In [18]:
texts[0]


"AI 최신 동향과 정책 (2025년 기준)\n\n\n2025년 현재, 인공지능(AI)은 기술적 성숙기를 지나 실질적인 서비스와 산업 전반으로 확산되고 있다. 단순히 텍스트를 생성하거나 이미지를 인식하는 수준을 넘어서, 인간의 복잡한 작업을 대행하는 'Agentic AI(지능형 에이전트)'가 주목받고 있다. 이러한 AI는 사용자 지시에 따라 이메일을 작성하고 회의 일정을 조율하며, 업무 흐름 전반을 자동화하는 수준에 이르렀다.\n\nAI 모델은 점차 전문화되고 있으며, 텍스트·이미지·음성·영상 등 다양한 데이터를 동시에 처리할 수 있는 '멀티모달 AI'가 보편화되고 있다. 특히 의료, 금융, 법률 등 전문 영역에 특화된 AI가 등장함에 따라, 범용 AI를 넘는 깊이 있는 서비스를 가능하게 하고 있다."

In [19]:
# Document 리스트 기반 분할
docs = text_splitter.split_documents(data)

# 이제는 Document이므로 page_content 사용 가능
print(len(docs[0].page_content))
print(docs[0])  # Document 전체 출력

# ============================================================
# 5. 라인별 주석
# ============================================================

# split_text()
# → 결과: List[str] (문자열 리스트)

# split_documents()
# → 결과: List[Document] (LangChain 구조 객체)

# page_content
# → Document에서만 사용 가능

# ============================================================
# 6. 핵심 포인트
# ============================================================

# - str vs Document 타입 차이가 핵심 원인
# - RAG에서는 split_documents()가 표준
# - page_content 사용하려면 반드시 Document 유지해야 함

390
page_content='AI 최신 동향과 정책 (2025년 기준)


2025년 현재, 인공지능(AI)은 기술적 성숙기를 지나 실질적인 서비스와 산업 전반으로 확산되고 있다. 단순히 텍스트를 생성하거나 이미지를 인식하는 수준을 넘어서, 인간의 복잡한 작업을 대행하는 'Agentic AI(지능형 에이전트)'가 주목받고 있다. 이러한 AI는 사용자 지시에 따라 이메일을 작성하고 회의 일정을 조율하며, 업무 흐름 전반을 자동화하는 수준에 이르렀다.

AI 모델은 점차 전문화되고 있으며, 텍스트·이미지·음성·영상 등 다양한 데이터를 동시에 처리할 수 있는 '멀티모달 AI'가 보편화되고 있다. 특히 의료, 금융, 법률 등 전문 영역에 특화된 AI가 등장함에 따라, 범용 AI를 넘는 깊이 있는 서비스를 가능하게 하고 있다.' metadata={'source': 'AI 최신 동향과 정책.txt'}


In [20]:
print(len(docs[1].page_content))
print(docs[1])


497
page_content='또한, AI를 클라우드 서버가 아닌 스마트폰이나 IoT 기기 등 ‘단말기’에서 직접 작동시키는 '온디바이스 AI'와 '엣지 컴퓨팅' 기술이 확산되고 있다. 이 기술은 속도와 프라이버시 측면에서 장점을 갖고 있으며, 자율주행, 웨어러블, 산업 로봇 등에 응용되고 있다.

비전문가도 AI를 쉽게 사용할 수 있도록 지원하는 ‘노코드’ 및 ‘로우코드’ 도구도 활발히 개발되고 있다. 이는 기업과 일반 사용자 모두가 AI를 일상과 업무에 손쉽게 접목할 수 있도록 돕고 있다.

이와 함께, AI가 소모하는 막대한 연산 자원과 에너지 사용 문제에 대한 경각심도 커지고 있다. AI 학습에 드는 전력량과 탄소 배출량을 줄이기 위한 ‘그린 AI’, 즉 지속 가능한 AI 개발이 중요한 화두로 떠오르고 있다.

AI는 이제 과학적 연구의 도구로도 자리 잡고 있다. 신약 개발, 단백질 구조 예측, 기후 변화 분석 등 복잡한 과학적 문제 해결에 AI가 활용되며, 연구 속도와 정확도가 크게 향상되고 있다.' metadata={'source': 'AI 최신 동향과 정책.txt'}


In [ ]:
"""
TextLoader
   ↓
TextSplitter (docs)
   ↓
Embedding (OpenAI)
   ↓
FAISS Vector DB
   ↓
Similarity Search
"""

In [21]:
# ============================================================
# 1. 상세 설명
# ============================================================

# 이 코드는 LangChain 기반 RAG 구조에서
# Document를 Embedding으로 변환하고 FAISS 벡터 DB에 저장하는 과정이다.
#
# 전체 흐름:
#
# Document (docs)
#   → Embedding 모델 (OpenAI)
#   → Vector DB (FAISS)
#   → 유사도 검색 기반 Retrieval
#
# 핵심 개념:
#
# - Embedding
#   → 텍스트를 숫자 벡터로 변환
#
# - FAISS
#   → 벡터 기반 유사도 검색 DB
#
# - similarity search
#   → 질문과 가장 가까운 문서 검색

# ============================================================
# 2. 패키지 설치 (Colab용)
# ============================================================

!pip install -q faiss-cpu langchain-openai langchain-community

# ============================================================
# 3. Import
# ============================================================

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# ============================================================
# 4. Embedding 모델 생성
# ============================================================

# OpenAI Embedding 모델 사용
# text → vector 변환
embeddings = OpenAIEmbeddings()

# ============================================================
# 5. Vector DB 생성 (FAISS)
# ============================================================

# docs: RecursiveCharacterTextSplitter로 생성된 Document 리스트
vector_db = FAISS.from_documents(
    docs,         # 문서 리스트
    embeddings    # 임베딩 모델
)

# ============================================================
# 6. 검색 테스트
# ============================================================

query = "이 문서에서 핵심 내용은 무엇인가?"

# 유사도 기반 검색 (Top-K = 3)
result_docs = vector_db.similarity_search(query, k=3)

# 결과 출력
print(result_docs[0].page_content)

미국은 상대적으로 자유로운 규제 정책을 유지하고 있다. 연방 차원에서는 가이드라인 중심이지만, 일부 주에서는 챗봇 표시 의무화, 투명성 확보 등을 법제화하고 있다. 반면, 중국은 강력한 중앙 통제를 통해 AI 기술을 관리하며, 사회적 통제와 국가안보 중심의 정책을 펼치고 있다.

일본, 싱가포르 등은 윤리와 산업 진흥을 함께 고려한 중간 지대의 정책을 추진하고 있으며, 국제기구(OECD, UNESCO 등)도 인간 중심의 AI 개발을 위한 국제적 윤리 기준을 마련하고 있다.


결론적으로, AI는 기술적, 산업적, 사회적 측면에서 거대한 변화를 일으키고 있으며, 이에 대한 정책적 대응 역시 빠르게 진화하고 있다. 기술의 발전을 뒷받침하는 동시에, 인간 중심의 AI 활용이 가능하도록 제도적 장치를 마련하는 것이 각국의 주요 과제가 되고 있다. 한국을 포함한 모든 국가는 이 균형점을 어떻게 설계할 것인가를 놓고 치열한 논의를 이어가고 있다.


# 실무 RAG 기본 아키텍처

In [22]:
# ============================================================
# 1. 상세 설명
# ============================================================

# 이 코드는 LCEL(LangChain Expression Language)을 사용하여
# FAISS Vector DB 기반 RAG 체인을 구성하는 구조이다.
#
# 전체 흐름:
#
# Question
#   ↓
# Retriever (FAISS)
#   ↓
# Context (유사 문서)
#   ↓
# PromptTemplate
#   ↓
# ChatOpenAI
#   ↓
# StrOutputParser
#   ↓
# Final Answer
#
# 핵심 개념:
#
# - retriever: Vector DB에서 관련 문서 검색
# - RunnablePassthrough: 입력 전달
# - LCEL pipe(|): 체인 연결 방식
# - StrOutputParser: AIMessage → 문자열 변환

# ============================================================
# 2. 패키지 Import
# ============================================================

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ============================================================
# 3. Retriever 생성
# ============================================================

# FAISS → Retriever 변환
retriever = vector_db.as_retriever(search_kwargs={"k":3})

# ============================================================
# 4. Prompt Template 정의
# ============================================================

prompt = ChatPromptTemplate.from_template("""
다음 context만 사용해서 질문에 답변하세요.

context:
{context}

question:
{question}

답변:
""")

# ============================================================
# 5. LLM 생성
# ============================================================

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# ============================================================
# 6. LCEL RAG Chain 구성
# ============================================================

rag_chain = (
    {
        "context" : retriever,
        "question" : RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()

)

# ============================================================
# 7. 실행 테스트
# ============================================================

result = rag_chain.invoke("이 문서의 핵심 내용은 뭐야?")
print(result)

이 문서의 핵심 내용은 인공지능(AI)의 기술 발전과 이에 대한 각국의 정책 및 규제 대응에 관한 것이다. 미국은 상대적으로 자유로운 규제 정책을 유지하고 있으며, 중국은 강력한 중앙 통제를 통해 AI 기술을 관리하고 있다. 일본과 싱가포르는 윤리와 산업 진흥을 함께 고려한 중간 지대의 정책을 추진하고 있다. 한국은 2024년 말 '인공지능 기본법' 개정안을 통과시키고, AI 윤리 기준을 수립하는 데 주력하고 있으나, 규제가 산업 진흥 위주로 설계되어 위험을 충분히 통제하지 못할 수 있다는 우려도 존재한다. AI는 기술적, 산업적, 사회적 측면에서 큰 변화를 일으키고 있으며, 각국은 인간 중심의 AI 활용을 위한 제도적 장치를 마련하는 데 집중하고 있다.


In [23]:
# ==============================================================================
# LCEL + FAISS 기반 RAG 시스템
#
# 시나리오
#
# BookFinder AI 서비스에서는
# 사용자가 원하는 도서의 종류를 입력하면
#
# 1. FAISS Vector DB에서 관련 도서를 검색하고
# 2. 검색된 도서 정보를 Context로 구성한 후
# 3. ChatOpenAI가 Context를 기반으로 도서를 추천한다.
#
# ==============================================================================
#
# 전체 실행 흐름
#
#               사용자 질문(User Question)
#                         │
#                         ▼
#               FAISS Retriever 검색
#                         │
#                         ▼
#               관련 Document 검색
#                         │
#                         ▼
#                 format_docs()
#                         │
#                         ▼
#                 Context(String)
#                         │
#                         ▼
#               ChatPromptTemplate
#                         │
#                         ▼
#                   ChatOpenAI
#                         │
#                         ▼
#                StrOutputParser
#                         │
#                         ▼
#                  최종 추천 결과
#
# ==============================================================================


# ==============================================================================
# 1. 라이브러리 Import
# ==============================================================================
#
# 역할
#
# LangChain에서 제공하는
# Prompt, OutputParser, Runnable 등을 불러온다.
#
# ==============================================================================

# Chat Prompt를 생성하기 위한 클래스
from langchain_core.prompts import ChatPromptTemplate

# AIMessage → 문자열(String)로 변환하는 Output Parser
from langchain_core.output_parsers import StrOutputParser

# LCEL(Runnable) 관련 클래스
from langchain_core.runnables import (
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)

# OpenAI Chat Model
from langchain_openai import ChatOpenAI

# OpenAI Embedding Model
from langchain_openai import OpenAIEmbeddings

# Vector Database
from langchain_community.vectorstores import FAISS


# ==============================================================================
# 2. ChatOpenAI 생성
# ==============================================================================
#
# 역할
#
# 검색된 Context와
# 사용자의 질문을 이용하여
# 최종 답변을 생성한다.
#
# 실행 흐름
#
# Prompt
#    │
#    ▼
# ChatOpenAI
#    │
#    ▼
# AI Answer
#
# ==============================================================================

# ChatOpenAI 객체 생성
llm = ChatOpenAI(
    model = "gpt-4o-mini"
)


# ==============================================================================
# 3. 원본 문서(Document)
# ==============================================================================
#
# 역할
#
# 실제 서비스에서는
#
# PDF
# Word
# Database
# CSV
# Excel
#
# 등의 데이터를 사용한다.
#
# 본 예제에서는 학습을 위해
# 간단한 문자열 리스트를 사용한다.
#
# ==============================================================================

docs = [

    # 첫 번째 문서
    "파이썬 코딩의 기술은 실무 중심의 Python 프로그래밍을 학습하기 좋은 도서이다.",

    # 두 번째 문서
    "혼자 공부하는 머신러닝+딥러닝은 머신러닝 입문자를 위한 대표적인 도서이다.",

    # 세 번째 문서
    "LLM 엔지니어링은 생성형 AI 시스템 구축 방법을 설명하는 실전 도서이다.",

]


# ==============================================================================
# 4. Embedding Model 생성
# ==============================================================================
#
# 역할
#
# 텍스트를 숫자(Vector)로 변환한다.
#
# 사람이 이해하는
#
# "머신러닝"
#
# 이라는 단어를
#
# AI가 계산할 수 있는
#
# [0.1234, -0.52, 0.83 ...]
#
# 와 같은 벡터로 변환한다.
#
# 이렇게 변환된 벡터를 이용하여
# 의미 기반 검색(Semantic Search)을 수행한다.
#
# ==============================================================================

# OpenAI Embedding 객체 생성
embeddings = OpenAIEmbeddings()


# ==============================================================================
# 5. FAISS Vector Database 생성
# ==============================================================================
#
# 역할
#
# Embedding된 문서를
# FAISS Vector Database에 저장한다.
#
# 이후 Retriever가
# 의미가 가장 유사한 문서를 검색하게 된다.
#
# 실행 흐름
#
# docs
#   │
#   ▼
# Embedding
#   │
#   ▼
# FAISS Vector DB
#
# ==============================================================================

# 문자열 문서를
# Embedding한 후
# FAISS Vector Database를 생성한다.

vectorstore = FAISS.from_texts(
    texts = docs,
    embedding = embeddings
)


# ==============================================================================
# 6. Retriever 생성
# ==============================================================================
#
# 역할
#
# Retriever는
#
# "사용자의 질문과 가장 유사한 문서를 검색하는 객체"
#
# 이다.
#
# Vector Database 자체는
# 검색 기능만 제공한다.
#
# Retriever는
#
# invoke()
#
# 를 통해
#
# 질문 → 문서 검색
#
# 기능을 수행한다.
#
# 실행 흐름
#
# Question
#      │
#      ▼
# Retriever
#      │
#      ▼
# List[Document]
#
# ==============================================================================

# VectorStore를 Retriever 형태로 변환한다.

retriever = vectorstore.as_retriever()

# ==============================================================================
# 7. Document → String 변환 함수
# ==============================================================================
#
# 역할
#
# Retriever의 반환값은
#
# List[Document]
#
# 형태이다.
#
# PromptTemplate은
#
# 문자열(String)
#
# 만 입력받을 수 있다.
#
# 따라서
#
# Document 객체를
#
# 하나의 Context 문자열로 변환해야 한다.
#
# 입력
#
# [
#   Document(...),
#   Document(...),
#   Document(...)
# ]
#
#
# 출력
#
# 문서1
#
# 문서2
#
# 문서3
#
# ==============================================================================

def format_docs(docs):
      return "\n\n".join(
          doc.page_content
          for doc in docs
      )

# ==============================================================================
# 8. ChatPromptTemplate 생성
# ==============================================================================
#
# 역할
#
# Retriever가 검색한 Context와
# 사용자가 입력한 Question을 하나의 Prompt로 구성한다.
#
# PromptTemplate은
#
# context
# question
#
# 두 개의 입력값을 받아
#
# 하나의 Prompt를 생성한다.
#
# 실행 흐름
#
# Context(String)
#        │
#        │
#        ▼
#
# Question
#        │
#        │
#        ▼
#
# ChatPromptTemplate
#        │
#        ▼
# Prompt 생성
#
# ==============================================================================

# ChatPromptTemplate 생성
prompt = ChatPromptTemplate.from_messages(

    [
        (
            "system",
            "당신은 도서 추천 전문가이다. 반드시 제공된 Context를 기반으로 답변하라."
        ),
        (
            "user",
            "Context\n{context}\n\nQuestion\n{question}"
        ),
    ]

)


# ==============================================================================
# 9. Retriever Chain 생성
# ==============================================================================
#
# 역할
#
# Retriever의 반환값은
#
# List[Document]
#
# 이다.
#
# Prompt는
#
# String
#
# 만 입력받을 수 있다.
#
# 따라서
#
# Retriever
#
# ↓
#
# RunnableLambda
#
# ↓
#
# Context(String)
#
# 형태로 변환해야 한다.
#
#
# 실행 흐름
#
# User Question
#        │
#        ▼
# Retriever
#        │
#        ▼
# List[Document]
#        │
#        ▼
# RunnableLambda
#        │
#        ▼
# Context(String)
#
# ==============================================================================

retriever_chain = (
    retriever
    | RunnableLambda(format_docs)
)



# ==============================================================================
# 10. LCEL 기반 RAG Chain
# ==============================================================================
#
# 역할
#
# RunnableParallel을 이용하여
#
# context
#
# question
#
# 두 개의 입력을 동시에 생성한다.
#
#
# 전체 실행 흐름
#
#
#                    User Question
#                          │
#          ┌───────┴───── ──┐
#          │                               │
#          ▼                               ▼
#
#   Retriever Chain              RunnablePassthrough
#
#          │                               │
#          ▼                               ▼
#
#   Context(String)                 User Question
#
#          └───────┬───────┘
#                          ▼
#
#                 ChatPromptTemplate
#                          │
#                          ▼
#
#                    ChatOpenAI
#                          │
#                          ▼
#
#                 StrOutputParser
#                          │
#                          ▼
#
#                    Final Answer
#
# ==============================================================================

rag_chain = (
    RunnableParallel(
        context = retriever_chain,
        question = RunnablePassthrough(),
    )
    | prompt
    | llm
    | StrOutputParser()
)




# ==============================================================================
# 11. RAG 실행
# ==============================================================================
#
# invoke()
#
# LCEL Chain을 실행한다.
#
#
# 입력
#
# "AI 관련 책 추천해줘."
#
#
# 실행 흐름
#
# Question
#
# ↓
#
# Retriever
#
# ↓
#
# Context
#
# ↓
#
# Prompt
#
# ↓
#
# ChatOpenAI
#
# ↓
#
# String
#
# ==============================================================================

# LCEL Chain 실행
result = rag_chain.invoke(
    "AI 관련 책을 추천해 주세요."
)


# ==============================================================================
# 12. 결과 출력
# ==============================================================================
#
# 역할
#
# 최종 생성된 문자열을 출력한다.
#
# ==============================================================================

print(result)

AI 관련 책으로는 다음 두 권을 추천드립니다:

1. **LLM 엔지니어링** - 이 책은 생성형 AI 시스템을 구축하는 방법을 실전 중심으로 설명하고 있어, 보다 깊은 이해와 실제 적용을 원하는 분들에게 적합합니다.

2. **혼자 공부하는 머신러닝+딥러닝** - 머신러닝과 딥러닝의 기초를 탄탄하게 다지고 싶은 입문자에게 적합한 도서입니다. 이 책을 통해 이론적 내용과 함께 실습을 통해 학습할 수 있습니다.

이 두 권의 책을 통해 AI의 기초부터 실전 적용까지 폭넓은 지식을 습득할 수 있을 것입니다.


RAG + LCEL 기반 AI 여행 플래너 서비스

In [ ]:
"""

시스템 구조

User Question
    ↓
FAISS Retriever
    ↓
Top-k 여행 문서
    ↓
PromptTemplate (context + question)
    ↓
ChatOpenAI
    ↓
StrOutputParser
    ↓
Final Answer

"""

In [ ]:
# ============================================================
# TO DO 문제: LCEL + FAISS 기반 RAG 체인 구조 완성하기
# 실행 X, 구조적 개념 이해
#
# 시나리오:
# TravelMind AI 서비스에서 사용자가 여행 질문을 하면
# FAISS가 관련 문서를 검색하고, LLM이 이를 기반으로
# 3일 여행 계획을 생성하는 RAG 시스템을 만든다.
#
# 목표:
# - FAISS Retriever
# - ChatPromptTemplate
# - ChatOpenAI
# - StrOutputParser
# 를 LCEL로 연결하여 완성하시오.
# ============================================================

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# ------------------------------------------------------------
# 1. LLM 모델 정의
# ------------------------------------------------------------
llm = ChatOpenAI()

# ------------------------------------------------------------
# 2. Prompt Template (TO DO)
# ------------------------------------------------------------
prompt = ChatPromptTemplate.from_messages([
    # system 역할: 여행 전문가로 설정하시오
    ("system",  "이 시스템은 여행 전문가이다. 주어진 context를 기반으로 여행계획을 생성한다."),

    # user 입력: context + question을 사용
    ("user",  "context : {context}\n\nquestion : {question}")
])

# ------------------------------------------------------------
# 3. Retriever (FAISS 기반이라고 가정)
# ------------------------------------------------------------
# vectorstore.as_retriever() 형태라고 가정
retriever = vectorstore.as_retriever()

# ------------------------------------------------------------
# 4. LCEL RAG Chain 구성 (TO DO)
# ------------------------------------------------------------
# 흐름:
# question → retriever → context → prompt → llm → string output

chain = (
    {
        "context" : retriever,    # retriever 결과 연결
        "question" : lambda x : x["question"] 사용자 입력 그대로 전달
    }
    prompt | llm | StrOutputParser()
)

# ------------------------------------------------------------
# 5. 실행 테스트
# ------------------------------------------------------------
result = chain.invoke({
    "question": "한국에서 3일 동안 혼자 여행하기 좋은 도시 2곳과 일정 추천해줘"
})

print(result)

# ============================================================
# TODO 핵심 체크 포인트
# 1. retriever는 FAISS 기반으로 구성되어야 함
# 2. LCEL에서 dict 입력 → prompt mapping 이해
# 3. context는 retriever 결과로 들어가야 함
# 4. StrOutputParser로 문자열 출력 변환
# ============================================================

문제) LCEL + FAISS 기반 RAG Chain 완성

In [ ]:
"""
다음의 표준 패턴에 맞는 문제를 작성하시오.

  Question
    │
    ▼
RunnableParallel
 ├───────────────┐
 │                              │
 ▼                              ▼
Retriever                RunnablePassthrough
 │                              │
 ▼                              ▼
RunnableLambda(format_docs)
 └──────┬────────┘
               ▼
        ChatPromptTemplate
               ▼
            ChatOpenAI
               ▼
        StrOutputParser


"""

In [ ]:
# ============================================================
# TO DO
# LCEL + FAISS 기반 RAG Chain 완성
#
# 시나리오
#
# TravelMind AI에서는
# 사용자의 여행 질문을 입력받아
#
# 1. FAISS에서 관련 문서를 검색하고
# 2. 검색된 문서를 하나의 Context로 합친 후
# 3. LLM에게 전달하여
# 4. 여행 계획을 생성하는 AI를 개발하고 있다.
#
# 아래 TODO를 완성하여
# 최신 LCEL 기반 RAG를 구현하시오.
# ============================================================

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
    RunnableLambda,
)

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# ------------------------------------------------------------
# 1. LLM
# ------------------------------------------------------------
llm = ChatOpenAI(model="gpt-4.1-mini")

# ------------------------------------------------------------
# 2. Documents
# ------------------------------------------------------------
docs = [
    "제주도는 자연경관이 아름답고 혼자 여행하기 좋은 지역이다.",
    "경주는 역사 유적 중심 여행지이다.",
    "부산은 해운대와 광안리가 유명하다."
]

# ------------------------------------------------------------
# TO DO 1
# OpenAI Embedding 객체를 생성하시오.
# ------------------------------------------------------------
embeddings =

# ------------------------------------------------------------
# TO DO 2
# docs와 embeddings를 이용하여 FAISS VectorStore를 생성하시오.
# ------------------------------------------------------------
vectorstore =

# ------------------------------------------------------------
# TO DO 3
# VectorStore로부터 Retriever를 생성하시오.
# ------------------------------------------------------------
retriever =

# ------------------------------------------------------------
# TO DO 4
# 검색된 Document 리스트를 하나의 문자열(Context)로
# 변환하는 함수를 완성하시오.
# Hint)
# Document.page_content 사용
# ------------------------------------------------------------
def format_docs(docs):



# ------------------------------------------------------------
# Prompt
# ------------------------------------------------------------
prompt = ChatPromptTemplate.from_messages(
[
    (
        "system",
        "당신은 여행 전문가이다. 반드시 Context 기반으로 답변한다."
    ),
    (
        "user",
        "Context\n{context}\n\nQuestion\n{question}"
    ),
]
)

# ------------------------------------------------------------
# TO DO 5
# Retriever와 format_docs를 연결하여
# Retriever Chain을 구성하시오.
#
# Hint)
# retriever |
# RunnableLambda(...)
# ------------------------------------------------------------
retriever_chain =



# ------------------------------------------------------------
# TO DO 6
# RunnableParallel을 이용하여
#
# context
# question
#
# 을 생성하는 LCEL 체인을 작성하시오.
#
# Hint)
# context = retriever_chain
# question = RunnablePassthrough()
# ------------------------------------------------------------
rag_chain = (




)

# ------------------------------------------------------------
# 실행
# ------------------------------------------------------------
result = rag_chain.invoke(
    "한국에서 2박3일 혼자 여행하기 좋은 곳 추천해줘."
)

print(result)

SyntaxError: invalid syntax (377000338.py, line 49)